In [33]:
%pip install pandas numpy statsmodels

Note: you may need to restart the kernel to use updated packages.


In [34]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

In [51]:
combine = pd.read_csv("2025 Combine Results.csv")
combine.columns = combine.columns.str.strip()

print(combine.shape)
combine.head()

(329, 15)


,Player,Pos,School,College,Ht,Wt,40yd,Vertical,Bench,Broad Jump,3Cone,Shuttle,Drafted (tm/rnd/yr),Player-additional,Drafted
0,BJ Adams,CB,Central Florida,College Stats,2-Jun,182,4.53,32.5,NaN,117.0,NaN,NaN,NaN,AdamBr00,0
1,Tommy Akingbesote,DT,Maryland,College Stats,4-Jun,306,5.09,28.0,NaN,103.0,NaN,NaN,Dallas Cowboys / 7th / 247th pick / 2025,AkinTo00,1
2,Darius Alexander,DT,Toledo,College Stats,4-Jun,305,4.95,31.5,28.0,111.0,7.6,4.79,New York Giants / 3rd / 65th pick / 2025,AlexDa01,1
3,Zy Alexander,CB,LSU,College Stats,1-Jun,187,4.56,31.5,NaN,116.0,NaN,NaN,NaN,AlexZy00,0
4,LeQuint Allen,RB,Syracuse,College Stats,Jun-00,204,NaN,35.0,NaN,120.0,NaN,NaN,Jacksonville Jaguars / 7th / 236th pick / 2025,AlleLe01,1


In [50]:
combine["Pick"] = combine["Drafted (tm/rnd/yr)"].str.extract(
    r'(\d+)(?=th pick|st pick|nd pick|rd pick)'
).astype(float)

print(combine["Pick"].notna().sum(), "players drafted")

215 players drafted


In [53]:
df = combine.dropna(subset=["Pick"]).copy()
print(df.shape)

(215, 16)


In [55]:
features = [
    "40yd", "Vertical", "Bench", "Broad Jump",
    "3Cone", "Shuttle", "Ht", "Wt", "Pos"
]

model_df = df[["Pick"] + features].copy()

In [56]:
num_cols = ["40yd", "Vertical", "Bench", "Broad Jump", "3Cone", "Shuttle", "Ht", "Wt"]

for col in num_cols:
    model_df[col] = pd.to_numeric(model_df[col], errors="coerce")

In [57]:
for col in num_cols:
    model_df[col + "_missing"] = model_df[col].isna().astype(int)

In [58]:
model_df[num_cols] = model_df.groupby("Pos")[num_cols].transform(
    lambda x: x.fillna(x.mean())
)

In [59]:
model_df[num_cols] = model_df[num_cols].fillna(model_df[num_cols].mean())

In [60]:
model_df = pd.get_dummies(model_df, columns=["Pos"], drop_first=True)

In [61]:
corr = model_df.corr(numeric_only=True)

high_corr = corr["Pick"].abs().sort_values(ascending=False)
print(high_corr.head(15))

Pick                  1.000000
Bench_missing         0.122717
Pos_CB/WR             0.109841
Vertical_missing      0.106583
Vertical              0.101355
Pos_RB                0.099267
Pos_P                 0.096276
Wt                    0.093602
Pos_EDGE              0.084839
Broad Jump_missing    0.084453
40yd_missing          0.082751
Pos_OT                0.079379
Broad Jump            0.077712
Pos_CB                0.077235
Bench                 0.075679
Name: Pick, dtype: float64


In [62]:
# Drop redundant / weaker variables
model_df = model_df.drop(columns=["Broad Jump", "3Cone", "Shuttle"], errors="ignore")

In [75]:
# Replace inf with NaN
model_df = model_df.replace([np.inf, -np.inf], np.nan)

# Keep relevant columns
model_df = combine[[
    "Pick",
    "40yd",
    "Wt",
    "Vertical",
    "Bench",
    "Pos"
]].copy()

# Convert to numeric
for col in ["Pick", "40yd", "Wt", "Vertical", "Bench"]:
    model_df[col] = pd.to_numeric(model_df[col], errors="coerce")

# ONLY drop rows where Pick is missing (undrafted players)
model_df = model_df.dropna(subset=["Pick"])

print("Rows left:", len(model_df))

Rows left: 215


In [76]:
# Fill missing combine drills with median (don’t drop rows)
for col in ["40yd", "Vertical", "Bench"]:
    model_df[col] = model_df[col].fillna(model_df[col].median())

In [77]:
model_df = pd.get_dummies(model_df, columns=["Pos"], drop_first=True)

In [79]:
X = model_df.drop(columns="Pick")
y = model_df["Pick"]

# 🔥 THIS LINE FIXES YOUR ERROR
X = X.astype(float)

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                   Pick   R-squared:                       0.120
Model:                            OLS   Adj. R-squared:                  0.039
Method:                 Least Squares   F-statistic:                     1.483
Date:                Thu, 23 Apr 2026   Prob (F-statistic):             0.0990
Time:                        01:57:41   Log-Likelihood:                -1207.7
No. Observations:                 215   AIC:                             2453.
Df Residuals:                     196   BIC:                             2518.
Df Model:                          18                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const        512.3287    230.693      2.221      0.0

In [80]:
model_df["predicted_pick"] = model.fittedvalues

In [81]:
model_df["percentile"] = pd.qcut(model_df["Pick"], 10, labels=False)

In [82]:
def assign_group(p):
    if p == 0:
        return "Top 10%"
    elif p == 1:
        return "Next 10%"
    elif p == 2:
        return "Next 10% After That"
    else:
        return "Remaining 70%"

model_df["group"] = model_df["percentile"].apply(assign_group)

In [83]:
group_summary = model_df.groupby("group").apply(
    lambda x: pd.Series({
        "Count": len(x),
        "Avg Actual Pick": x["Pick"].mean(),
        "Avg Predicted Pick": x["predicted_pick"].mean(),
        "Mean Absolute Error": abs(x["Pick"] - x["predicted_pick"]).mean()
    })
).reset_index()

# Sort nicely
order = ["Top 10%", "Next 10%", "Next 10% After That", "Remaining 70%"]
group_summary["group"] = pd.Categorical(group_summary["group"], categories=order, ordered=True)

group_summary = group_summary.sort_values("group")

print(group_summary)

                 group  Count  Avg Actual Pick  Avg Predicted Pick  \
3              Top 10%   22.0        11.727273          103.560869   
0             Next 10%   21.0        34.000000          107.593611   
1  Next 10% After That   22.0        55.500000          110.444559   
2        Remaining 70%  150.0       151.706667          119.876098   

   Mean Absolute Error  
3            91.833596  
0            73.593611  
1            54.944559  
2            47.719033  
